[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S04/S04_05_poo_buenas_practicas.ipynb)

# S04_05 · (Opcional) Programación orientada a objetos básica, buenas prácticas y Git
**Sprint 4 · Python para IA · ⏱ 60 min · opcional**

## Qué vas a aprender
1. A definir **clases** con atributos y métodos, y a crear objetos.
2. A usar `@classmethod` para construir objetos desde un diccionario y **herencia** para especializar.
3. A aplicar buenas prácticas: PEP 8, docstrings, `if __name__ == "__main__"` y pruebas con `assert`.
4. A conocer los comandos básicos de **Git** para versionar tu reto.

## Contexto TurisData
En vez de pasar diccionarios sueltos, modelamos el negocio con **objetos**: una `Reserva` sabe calcular su
ingreso; una `ColeccionReservas` sabe calcular KPIs. Es el paso previo a organizar bien tu reto.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import csv

## 1 · Clases y objetos
Una **clase** es un molde; un **objeto** es una pieza hecha con ese molde. `__init__` prepara el objeto,
`self` es "el propio objeto", y los **métodos** son funciones de la clase. (Comparación con Java: `self`
equivale a `this`, pero hay que escribirlo explícitamente; no hay `public`/`private`: se usa la convención
`_nombre` para "uso interno".)

In [ ]:
class Reserva:
    """Una reserva de TurisData."""

    def __init__(self, id_reserva, canal, noches, precio_noche, cancelada=False):
        self.id_reserva = id_reserva
        self.canal = canal
        self.noches = noches
        self.precio_noche = precio_noche
        self.cancelada = cancelada

    def ingreso(self):
        """Ingreso bruto de la reserva (noches por precio)."""
        return self.noches * self.precio_noche

    def __repr__(self):
        return f"Reserva({self.id_reserva}, {self.canal}, {self.noches} noches)"


r = Reserva(5760, "empresa", 3, 89.24, cancelada=True)
print(r, r.ingreso())

### Ejercicio 1 · Añadir un método
Añade a la clase `ReservaMejorada` (definida abajo copiando la anterior) el método `es_larga(self, umbral=7)`
que devuelva `True` si la reserva tiene `umbral` o más noches. Completa el cuerpo del método.

In [ ]:
# TODO: dentro de la clase, un método que compare self.noches con el umbral
...
rm = ReservaMejorada(1, "directo", 7, 100.0)
assert rm.es_larga() is True, "7 noches con umbral 7 es una estancia larga"
assert ReservaMejorada(2, "directo", 6, 100.0).es_larga() is False, "6 noches con umbral 7 no es larga"
assert ReservaMejorada(3, "directo", 4, 100.0).es_larga(umbral=4) is True, "El umbral se puede cambiar"
assert rm.ingreso() == 700.0, "ReservaMejorada hereda ingreso() de Reserva"
print("Correcto")

## 2 · `@classmethod`: construir objetos desde un diccionario
Cuando leemos un CSV, cada fila llega como diccionario de textos. Un **método de clase** actúa como
"constructor alternativo": recibe la clase (`cls`) y devuelve un objeto.

In [ ]:
class ReservaCSV(Reserva):
    """Reserva que sabe construirse desde una fila del CSV."""

    @classmethod
    def desde_fila(cls, fila):
        """Crea una reserva a partir de una fila (diccionario de textos) del CSV de TurisData."""
        return cls(
            id_reserva=int(fila["id_reserva"]),
            canal=fila["canal"],
            noches=int(fila["noches"]),
            precio_noche=float(fila["precio_noche"]),
            cancelada=fila["cancelada"] == "1",
        )


fila = {"id_reserva": "5760", "canal": "empresa", "noches": "3", "precio_noche": "89.24", "cancelada": "1"}
print(ReservaCSV.desde_fila(fila))

### Ejercicio 2 · Herencia y sobrescritura
Las reservas de **grupo** tienen un **10 % de descuento** sobre el ingreso. Crea la clase `ReservaGrupo`
heredando de `Reserva` que sobrescriba `ingreso()` devolviendo el importe ya rebajado un 10 %
(usa `super().ingreso()`).

In [ ]:
# TODO: class ReservaGrupo(Reserva) con un método ingreso que llame a super().ingreso() y multiplique por 0.9
...
g = ReservaGrupo(9, "agencia", 5, 100.0)
assert g.ingreso() == 450.0, f"5 x 100 con un 10 % de descuento son 450.0 y tienes {g.ingreso()}"
assert isinstance(g, Reserva), "ReservaGrupo debe heredar de Reserva"
print("Correcto:", g.ingreso())

## 3 · Una clase que agrupa objetos
Una `ColeccionReservas` guarda una lista de reservas y ofrece métodos de análisis (nuestros futuros KPIs).

In [ ]:
class ColeccionReservas:
    """Conjunto de reservas con métodos de análisis."""

    def __init__(self, reservas):
        self.reservas = list(reservas)

    def __len__(self):
        return len(self.reservas)

    def canceladas(self):
        """Número de reservas canceladas."""
        return sum(1 for r in self.reservas if r.cancelada)


demo = ColeccionReservas([Reserva(1, "directo", 3, 80.0), Reserva(2, "agencia", 2, 150.0, True)])
print(len(demo), demo.canceladas())

### Ejercicio 3 · Métodos de KPI
Añade a `ColeccionKPI` (que hereda de `ColeccionReservas`) dos métodos:
- `tasa_cancelacion()` → porcentaje de canceladas con 1 decimal (`0.0` si no hay reservas).
- `ingreso_medio_confirmadas()` → ingreso medio (2 decimales) de las reservas **no** canceladas (`0.0` si no hay ninguna).

In [ ]:
# TODO: reutiliza self.canceladas() y len(self); filtra las no canceladas para el ingreso medio y protege la división entre cero
...
prueba = ColeccionKPI([Reserva(1, "directo", 3, 80.0), Reserva(2, "agencia", 2, 150.0, True),
                       Reserva(3, "directo", 4, 110.0), Reserva(4, "web", 1, 70.0, True)])
assert prueba.tasa_cancelacion() == 50.0, f"2 de 4 canceladas es 50.0 % y tienes {prueba.tasa_cancelacion()}"
assert prueba.ingreso_medio_confirmadas() == 340.0, f"(240+440)/2 = 340.0 y tienes {prueba.ingreso_medio_confirmadas()}"
assert ColeccionKPI([]).tasa_cancelacion() == 0.0, "Una colección vacía debe dar 0.0, no error"
assert ColeccionKPI([]).ingreso_medio_confirmadas() == 0.0, "Sin reservas confirmadas debe dar 0.0"
print("Correcto")

## 4 · Buenas prácticas de código
| Práctica | Ejemplo |
|---|---|
| Nombres claros (PEP 8) | `tasa_cancelacion`, clases en `MayusculasIniciales`, constantes en `MAYUSCULAS` |
| Docstring en cada función/clase | una frase que diga qué hace y qué devuelve |
| Funciones pequeñas con una sola tarea | leer, calcular y exportar son funciones distintas |
| No repetir código | si copias y pegas dos veces, haz una función |
| Errores claros | mensajes que digan qué ha pasado y cómo arreglarlo |
| Pruebas | `assert` sobre casos conocidos (también los casos límite: lista vacía, división entre cero) |
| Punto de entrada | `if __name__ == "__main__":` para que el archivo sirva como script **y** como módulo |

Estructura mínima de un repositorio para el reto:
```
herramientas_internas/
├── README.md          # qué hace, cómo se ejecuta, cómo se prueban
├── kpis.py            # el script (funciones + main)
├── test_kpis.py       # pruebas con assert
└── datos/reservas_turisdata.csv
```

## 5 · Git básico (se practica en el terminal o en VS Code; aquí solo se explica)
| Orden | Para qué |
|---|---|
| `git init` | Convertir la carpeta en repositorio |
| `git status` | Ver qué ha cambiado |
| `git add kpis.py test_kpis.py README.md` | Preparar archivos para el siguiente registro |
| `git commit -m "Lee el CSV y calcula la tasa de cancelación"` | Guardar una versión con un mensaje claro |
| `git log --oneline` | Ver el historial |
| `git switch -c mejora-errores` | Crear una rama para probar sin romper lo que funciona |
| `git push` | Subir los commits al repositorio remoto (GitHub) |

Consejo: commits pequeños y frecuentes, con mensajes que digan **qué** cambia. No subas datos sensibles ni
claves. Un archivo `.gitignore` evita subir `__pycache__/` y archivos temporales.

## 6 · Mini-reto integrador: el patrón del script
Usa las clases de este cuaderno con los **datos reales**. Completa la función `cargar_coleccion(ruta)` que lea
el CSV con `csv.DictReader`, cree objetos con `ReservaCSV.desde_fila` y devuelva una `ColeccionKPI`.
Después calculamos `kpis_reales`, un diccionario con la tasa de cancelación y el ingreso medio.

In [ ]:
ruta_reservas = dato("reservas_turisdata.csv")


class ReservaKPI(ReservaCSV):
    """Reserva desde CSV (alias para la colección de KPIs)."""


# TODO: abre el CSV con csv.DictReader, crea cada objeto con ReservaKPI.desde_fila y pasa la lista a ColeccionKPI
...
coleccion = cargar_coleccion(ruta_reservas)
kpis_reales = {"reservas": len(coleccion), "tasa_cancelacion": coleccion.tasa_cancelacion(),
               "ingreso_medio_confirmadas": coleccion.ingreso_medio_confirmadas()}
assert kpis_reales["reservas"] == 6000, f"Debe cargar las 6000 reservas y has cargado {kpis_reales['reservas']}"
assert kpis_reales["tasa_cancelacion"] == 45.9, f"La tasa de cancelación global es 45.9 % y tienes {kpis_reales['tasa_cancelacion']}"
assert kpis_reales["ingreso_medio_confirmadas"] == 576.04, f"El ingreso medio es 576.04 y tienes {kpis_reales['ingreso_medio_confirmadas']}"
print("Correcto:", kpis_reales)

## 7 · Para reflexionar
1. ¿Qué ventaja tiene que `Reserva` calcule su propio ingreso en lugar de repetir `noches * precio` por todo el código?
2. ¿Cuándo es preferible una función suelta y cuándo una clase?
3. ¿Qué mensaje de commit ayudaría a tu yo del futuro?

## 8 · Qué has aprendido
- Clases, atributos, métodos, `__init__`, `__repr__`, herencia y `@classmethod`.
- Buenas prácticas: PEP 8, docstrings, funciones pequeñas, pruebas con `assert`.
- Estructura de repositorio y comandos básicos de Git.
- Ya tienes todas las piezas para el reto **«Herramientas internas»**.